In [1]:
import matplotlib.pyplot as plt
from dstools.dynamic_spectrum import DynamicSpectrum, LightCurve
from dstools.plotting import plot_ds, plot_lightcurve
import os
from pathlib import Path
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from matplotlib.patches import Rectangle
import astropy.constants as c
import astropy.units as u
from scipy.optimize import curve_fit

From Callingham 2025:

$$ \frac{d\nu}{dt} = -\nu v_s / 2H$$

From my results, $\frac{d\nu}{dt} \approx$ -100 MHz/hr (-95 $\pm$ 10 MHz/hr)

In [2]:
dnu_dt = -100 * u.MHz / u.hr

If we assume that our burst came from plasma emission, we can estimate the velocity of the event causing the emission if we know the scale height. We measure v/H from our burst start, so if we can get an independent measure of H, we can estimate v. To calculate H, we can use the formula (Vedantham 2020):

$$ H = 6 \times 10^9 \: (T_c/10^6K) \: (R_*/R_{\odot})^2 \: (M_*/M_{\odot})^{-1}$$

Here, $T_c$ is the coronal temperature, which we can estimate if we know the quiescent xray luminosity of the star.

To get to $T_c$, we can use the scaling relation from Johnstone 2015,

$$ T_c = 0.11F_X^{0.26}$$

Here, $F_X$ is the flux in the 0.2-2.4 keV band. First, let's check that we can re-derive Joe's results to make sure we have units etc correct.

In [4]:
Fx_obs = 1.59e-12 * u.erg / u.s / u.cm**2
d = 40.8 * u.pc
R = 0.6 * u.Rsun

Lx = 4 * np.pi * d.to(u.cm)**2 * Fx_obs
Fx_surf = Lx / (4 * np.pi * (R.to(u.cm))**2)

Tc_MK = 0.11 * (Fx_surf.value)**0.26
Tc = Tc_MK * u.MK

In [112]:
Lx

<Quantity 3.16685665e+29 erg / s>

So we CAN reproduce these results given we get an xray flux for my star. Let's start with a simpler answer and try and estimate what the velocity would be for a solar-like corona (~2MK).

# Estimating using solar values

In [113]:
Tc_sun = 2 # in Tc/10^6

For approximation purposes, assume that HD 104919 is similar in mass and radius to the sun, so our scale height is just given by:

In [114]:
H = 6e9 * Tc_sun *u.cm

In [115]:
nu = 950* u.MHz

In [116]:
vs = dnu_dt * 2 * H / -nu

In [117]:
vs.to(u.m / u.s)

<Quantity 7017.54385965 m / s>

# Using HD 104919 data

In [6]:
h1 = 0.46
ct = 3.47e-01

In [9]:
xrs_flux = ((8.31 + 5.3 * h1) * 1e-12 * ct) * (u.erg / u.s / u.cm**2)
xrs_flux

<Quantity 3.729556e-12 erg / (s cm2)>

In [118]:
xmm_flux = 1.45e-15 * u.W / u.m**2

In [119]:
xmm_flux.to(u.erg / u.s / u.cm**2)

<Quantity 1.45e-12 erg / (s cm2)>

In [10]:
d = 100.9 * u.pc
R = 1.36 * u.Rsun

Lx = 4 * np.pi * d.to(u.cm)**2 * xrs_flux
Fx_surf = Lx / (4 * np.pi * (R.to(u.cm))**2)

Tc_MK = 0.11 * (Fx_surf.value)**0.26
Tc = Tc_MK * u.MK

In [11]:
Lx

<Quantity 4.54308037e+30 erg / s>

In [12]:
Tc

<Quantity 10.44743206 MK>

Note this is already a way higher Tc than the sun --> Active star!

In [18]:
H = 6e9 * (Tc.value) * R.value**2 * u.cm
nu = 950* u.MHz
vs = dnu_dt * 2 * H / -nu

In [19]:
vs.to(u.km / u.s)

<Quantity 67.80200117 km / s>

This is quite a low velocity, so this likely isn't plasma emission we are seeing. To confirm, we should check the escape velocity of the star

# Escape velocity

Need to consider both the gravitational force and the magnetic confinement to escape this star. Use the Alfven velocity as a proxy for a magnetic escape velocity.

In [20]:
M = 1 * u.Msun
v_esc_g = (2 * c.G * M / R)**(1/2)

In [21]:
v_esc_g.to(u.km / u.s)

<Quantity 529.65168588 km / s>

In [130]:
rho = M / R**3
B = 300 * u.G

In [133]:
v_esc_b = B * (c.mu0 * rho)**(-1/2)

In [134]:
v_esc_b.to(u.km / u.s)

<Quantity 0.00050559 km / s>

So essentially, the escape velocity is significantly higher than the calculated CME speed, which indicates to us that the event is likely ECMI, not plasma emission.